In [1]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
import warnings
warnings.filterwarnings("ignore")

builder = (
    SparkSession.builder
    .appName("delta-minio-test")
    .master("local[*]")
    .config(
        "spark.jars.packages",
        ",".join([
            "io.delta:delta-spark_2.12:3.2.0",
            "org.apache.hadoop:hadoop-aws:3.3.4",
            "com.amazonaws:aws-java-sdk-bundle:1.12.262",
        ])
    )
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "minio")
    .config("spark.hadoop.fs.s3a.secret.key", "minio123")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-b63d23e3-af60-44de-9b25-575dd45efcdc;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
downloading https://repo1.maven.org/maven2/io/delta/delta-spark_2.12/3.2.0/delta-spark_2.12-3.2.0.jar ...
	[SUCCESSFUL ] io.delta#delta-spark_2.12;3.2.0!delta-spark_2.12.jar (427ms)
downloading https://repo1.maven.org/maven2/io/delta/delta-storage/3.2.0/delta-storage-3.2.0.jar ...
	[SUCCESSFUL ] io.delta#delta-storage;3.2.0!delta-storage.jar (46ms)
downloading https://repo1.maven.org/maven2/org/antlr/antlr4-runtime/4.9.3/antlr4-runtime-4.9.3.jar ...
	[SUCCESSFUL ] org.antlr#antlr4-runtime;4.9.3!antlr4-runtime.jar (63ms)
:: resolution report :: resolve 886ms :: artifacts dl 539

In [2]:
user_silver_path = "s3a://lakehouse/silver_delta/user_events_clean"

df = spark.read.format("delta").load(user_silver_path)
df.printSchema()
df.count()

26/03/24 18:49:56 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


root
 |-- raw_json: string (nullable = true)
 |-- kafka_topic: string (nullable = true)
 |-- kafka_partition: integer (nullable = true)
 |-- kafka_offset: long (nullable = true)
 |-- kafka_timestamp: timestamp (nullable = true)
 |-- bronze_ingest_ts: timestamp (nullable = true)
 |-- event_id: string (nullable = true)
 |-- event_ts: timestamp (nullable = true)
 |-- event_type: string (nullable = true)
 |-- source: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- anonymous_id: string (nullable = true)
 |-- session_id: string (nullable = true)
 |-- page_url: string (nullable = true)
 |-- properties: struct (nullable = true)
 |    |-- item_id: string (nullable = true)
 |    |-- item_title: string (nullable = true)
 |    |-- backend_event_name: string (nullable = true)
 |    |-- message: string (nullable = true)
 |-- event_date: date (nullable = true)
 |-- canonical_user_key: string (nullable = true)
 |-- item_id: string (nullable = true)
 |-- item_title: string (nullabl

26/03/24 18:49:59 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

33

In [3]:
df.show()

+--------------------+-----------+---------------+------------+--------------------+--------------------+--------------------+--------------------+------------+--------+-------+--------------------+--------------------+--------------------+--------------------+----------+--------------------+--------+------------------+------------------+
|            raw_json|kafka_topic|kafka_partition|kafka_offset|     kafka_timestamp|    bronze_ingest_ts|            event_id|            event_ts|  event_type|  source|user_id|        anonymous_id|          session_id|            page_url|          properties|event_date|  canonical_user_key| item_id|        item_title|backend_event_name|
+--------------------+-----------+---------------+------------+--------------------+--------------------+--------------------+--------------------+------------+--------+-------+--------------------+--------------------+--------------------+--------------------+----------+--------------------+--------+----------------

In [7]:
from IPython.display import display

display(df.toPandas())

,raw_json,kafka_topic,kafka_partition,kafka_offset,kafka_timestamp,bronze_ingest_ts,event_id,event_ts,event_type,source,user_id,anonymous_id,session_id,page_url,properties,event_date,canonical_user_key,item_id,item_title,backend_event_name
0,"{""event_id"": ""evt_985da14f88ca49dd8545f9482d0d...",user_events,1,1,2026-03-24 18:16:06.440,2026-03-24 18:16:15.009,evt_985da14f88ca49dd8545f9482d0d1276,2026-03-24 18:16:06.423937,backend_log,backend,None,anon_a271676c602d4b24a09e438333abd2f4,sess_95e1566d91254e8cb50c59a030a3f16d,http://localhost:8091/,"(sku_1001, لپ‌تاپ ProBook X, order_check, stru...",2026-03-24,anon_a271676c602d4b24a09e438333abd2f4,sku_1001,لپ‌تاپ ProBook X,order_check
1,"{""event_id"": ""evt_f5a059c79d0c447d9d3245b6d5e5...",user_events,2,5,2026-03-24 18:17:20.994,2026-03-24 18:17:30.011,evt_f5a059c79d0c447d9d3245b6d5e5049a,2026-03-24 18:17:20.982030,backend_log,backend,None,anon_67b4d31fdf404cf1bac7c48b243ad133,sess_74c46fe908c84330a2493c0e89f6d609,http://localhost:8091/,"(sku_1001, لپ‌تاپ ProBook X, order_check, stru...",2026-03-24,anon_67b4d31fdf404cf1bac7c48b243ad133,sku_1001,لپ‌تاپ ProBook X,order_check
2,"{""event_id"": ""evt_073a1d0e-416b-486b-bf61-bfd2...",user_events,1,2,2026-03-24 18:16:07.124,2026-03-24 18:16:15.009,evt_073a1d0e-416b-486b-bf61-bfd2b94f9102,2026-03-24 18:16:07.122000,product_view,frontend,None,anon_a271676c602d4b24a09e438333abd2f4,sess_95e1566d91254e8cb50c59a030a3f16d,/,"(sku_1001, لپ‌تاپ ProBook X, None, None)",2026-03-24,anon_a271676c602d4b24a09e438333abd2f4,sku_1001,لپ‌تاپ ProBook X,None
3,"{""event_id"": ""evt_5a7ac872-4277-41fa-a05a-d898...",user_events,2,3,2026-03-24 02:38:34.969,2026-03-24 02:38:45.050,evt_5a7ac872-4277-41fa-a05a-d898930af909,2026-03-24 02:38:34.968000,product_view,frontend,None,anon_6e78abb16d4a40d0a0cfc080373771bb,sess_aae11e575898400f8f2b4cc09dfda910,/,"(sku_1001, لپ‌تاپ ProBook X, None, None)",2026-03-24,anon_6e78abb16d4a40d0a0cfc080373771bb,sku_1001,لپ‌تاپ ProBook X,None
4,"{""event_id"": ""evt_64c12ed4-3a64-4179-9109-47b7...",user_events,0,3,2026-03-24 02:39:44.508,2026-03-24 02:39:45.011,evt_64c12ed4-3a64-4179-9109-47b7be1765c3,2026-03-24 02:39:44.506000,product_view,frontend,None,anon_44278a99105a437a98a78606a953b0e8,sess_c470801c6e294322881676d6c68da90d,/,"(sku_1001, لپ‌تاپ ProBook X, None, None)",2026-03-24,anon_44278a99105a437a98a78606a953b0e8,sku_1001,لپ‌تاپ ProBook X,None
5,"{""event_id"": ""evt_25515588-b093-4de3-9068-96f2...",user_events,2,1,2026-03-24 18:16:10.022,2026-03-24 18:16:15.009,evt_25515588-b093-4de3-9068-96f27ed6fa7f,2026-03-24 18:16:10.019000,product_view,frontend,None,anon_f007f097e597433faae418cd521cbd06,sess_2b1ceb0e907c40db8a8cdc686fa5209b,/,"(sku_1003, هدفون AirSound Max, None, None)",2026-03-24,anon_f007f097e597433faae418cd521cbd06,sku_1003,هدفون AirSound Max,None
6,"{""event_id"": ""evt_161b7ade-bb4c-49e0-ba55-8b84...",user_events,2,4,2026-03-24 18:17:19.494,2026-03-24 18:17:30.011,evt_161b7ade-bb4c-49e0-ba55-8b8465506a4a,2026-03-24 18:17:19.491000,add_to_cart,frontend,None,anon_67b4d31fdf404cf1bac7c48b243ad133,sess_74c46fe908c84330a2493c0e89f6d609,/,"(sku_1001, لپ‌تاپ ProBook X, None, None)",2026-03-24,anon_67b4d31fdf404cf1bac7c48b243ad133,sku_1001,لپ‌تاپ ProBook X,None
7,"{""event_id"": ""evt_d7ac1757-9b36-4409-b8f8-fea5...",user_events,1,2,2026-03-24 02:37:13.263,2026-03-24 02:37:15.009,evt_d7ac1757-9b36-4409-b8f8-fea564c8efcf,2026-03-24 02:37:13.262000,add_to_cart,frontend,None,anon_d4208bf33dd541c6baeab48b6361e7cd,sess_5777975ee54b4447816ff565c337ad98,/,"(sku_1001, لپ‌تاپ ProBook X, None, None)",2026-03-24,anon_d4208bf33dd541c6baeab48b6361e7cd,sku_1001,لپ‌تاپ ProBook X,None
8,"{""event_id"": ""evt_6a50bf01-f193-4bd5-b96e-9565...",user_events,0,2,2026-03-24 02:39:43.998,2026-03-24 02:39:45.011,evt_6a50bf01-f193-4bd5-b96e-95657c9980b2,2026-03-24 02:39:43.996000,add_to_cart,frontend,None,anon_44278a99105a437a98a78606a953b0e8,sess_c470801c6e294322881676d6c68da90d,/,"(sku_1001, لپ‌تاپ ProBook X, None, None)",2026-03-24,anon_44278a99

In [4]:
from py4j.protocol import Py4JJavaError

quarantine_path = "s3a://lakehouse/silver_delta/user_events_quarantine"

try:
    dfq = spark.read.format("delta").load(quarantine_path)
    dfq.show(20, truncate=False)
    print("quarantine count:", dfq.count())
except Py4JJavaError as e:
    if "PATH_NOT_FOUND" in str(e):
        print("Quarantine table does not exist yet. No invalid records have been written.")
    else:
        raise

AnalysisException: [PATH_NOT_FOUND] Path does not exist: s3a://lakehouse/silver_delta/user_events_quarantine.